# Exercise 9.02 — Comparing the plotting and models interfaces

In [1]:
import numpy as np
import pandas as pd
from bokeh.io import output_notebook
output_notebook()

Loading BokehJS ...

In [2]:
dataset = pd.read_csv('../Datasets/world_population.csv', index_col=0)

In [3]:
dataset.head()

,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,1966,...,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016
Country Name,,,,,,,,,,,,,,,,,,,,,
Aruba,ABW,Population density (people per sq. km of land ...,EN.POP.DNST,NaN,307.972222,312.366667,314.983333,316.827778,318.666667,320.622222,...,562.322222,563.011111,563.422222,564.427778,566.311111,568.850000,571.783333,574.672222,577.161111,NaN
Andorra,AND,Population density (people per sq. km of land ...,EN.POP.DNST,NaN,30.587234,32.714894,34.914894,37.170213,39.470213,41.800000,...,180.591489,182.161702,181.859574,179.614894,175.161702,168.757447,161.493617,154.863830,149.942553,NaN
Afghanistan,AFG,Population density (people per sq. km of land ...,EN.POP.DNST,NaN,14.038148,14.312061,14.599692,14.901579,15.218206,15.545203,...,39.637202,40.634655,41.674005,42.830327,44.127634,45.533197,46.997059,48.444546,49.821649,NaN
Angola,AGO,Population density (people per sq. km of land ...,EN.POP.DNST,NaN,4.305195,4.384299,4.464433,4.544558,4.624228,4.703271,...,15.387749,15.915819,16.459536,17.020898,17.600302,18.196544,18.808215,19.433323,20.070565,NaN
Albania,ALB,Population density (people per sq. km of land ...,EN.POP.DNST,NaN,60.576642,62.456898,64.329234,66.209307,68.058066,69.874927,...,108.394781,107.566204,106.843759,106.314635,106.013869,105.848431,105.717226,105.607810,105.444051,NaN


In [4]:
from bokeh.plotting import figure, show

In [5]:
years = [year for year in dataset.columns if not year[0].isalpha()]
mean_pop_vals = [np.mean(dataset[year]) for year in years]
jp_vals = [dataset.loc[['Japan']][year] for year in years]

In [6]:
plot = figure(title='Global Mean Population Density compared to Japan',
              x_axis_label='Year', y_axis_label='Population Density')
plot.line(years, mean_pop_vals, line_width=2, legend_label='Global Mean')
plot.cross(years, jp_vals, legend_label='Japan', line_color='red')
plot.legend.location = 'bottom_right'
show(plot)

## Using the models interface

In [7]:
# importing the models dependencies
from bokeh.io import show
from bokeh.models.grids import Grid
from bokeh.models.plots import Plot
from bokeh.models.axes import LinearAxis
from bokeh.models.ranges import Range1d
from bokeh.models.glyphs import Line, Scatter
from bokeh.models.sources import ColumnDataSource
from bokeh.models.tickers import SingleIntervalTicker, YearsTicker
from bokeh.models.renderers import GlyphRenderer
from bokeh.models.annotations import Title, Legend, LegendItem

In [8]:
# defining the range for the x and y axis
extracted_mean_pop_vals = \
    [val for i, val in enumerate(mean_pop_vals) \
     if i not in [0, len(mean_pop_vals) - 1]]
extracted_jp_vals = \
    [jp_val['Japan'] for i, jp_val in enumerate(jp_vals) \
     if i not in [0, len(jp_vals) - 1]]
min_pop_density = min(extracted_mean_pop_vals)
min_jp_density = min(extracted_jp_vals)
min_y = int(min(min_pop_density, min_jp_density))
max_pop_density = max(extracted_mean_pop_vals)
max_jp_density = max(extracted_jp_vals)
max_y = int(max(max_jp_density, max_pop_density))
xdr = Range1d(int(years[0]), int(years[-1]))
ydr = Range1d(min_y, max_y)

In [9]:
axis_def = dict(axis_line_color='#222222', axis_line_width=1,
                major_tick_line_color='#222222',
                major_label_text_color='#222222',
                major_tick_line_width=1)

x_axis = LinearAxis(ticker=SingleIntervalTicker(interval=10),
                    axis_label='Year', **axis_def)
y_axis = LinearAxis(ticker=SingleIntervalTicker(interval=50),
                    axis_label='Population Density', **axis_def)

In [10]:
# creating the plot object
title = Title(align='left', text='Global Mean Population Density compared to Japan')
plot = Plot(x_range=xdr, y_range=ydr, width=650, height=600, title=title)

In [11]:
"""
error will be thrown because we are missing renderers that are created when adding elements
"""
show(plot)

In [12]:
# creating the data display
line_source = ColumnDataSource(dict(x=years, y=mean_pop_vals))
line_glyph = Line(x='x', y='y', line_color='#2678b2', line_width=2)

cross_source = ColumnDataSource(dict(x=years, y=jp_vals))
cross_glyph = Scatter(x='x', y='y', marker='cross', line_color='#fc1d26')

In [13]:
plot.add_layout(x_axis, 'below')
plot.add_layout(y_axis, 'left')
line_renderer = plot.add_glyph(line_source, line_glyph)
cross_renderer = plot.add_glyph(cross_source, cross_glyph)

In [14]:
show(plot)

In [15]:
legend_items = [LegendItem(label='Global Mean', renderers=[line_renderer]),
                LegendItem(label='Japan', renderers=[cross_renderer])]
legend = Legend(items=legend_items, location='bottom_right')

In [16]:
# creating the grid
x_grid = Grid(dimension=0, ticker=x_axis.ticker)
y_grid = Grid(dimension=1, ticker=y_axis.ticker)

In [17]:
plot.add_layout(legend)
plot.add_layout(x_grid)
plot.add_layout(y_grid)
show(plot)